# Monotonic Stack & Queue

A stack (or deque) kept in sorted order, where you **discard elements that can never again be the answer**. It's the O(n) key to a whole family of "nearest greater/smaller element" and "window max/min" problems that look quadratic at first. This is the technique the stacks-and-queues and sliding-window notebooks pointed forward to.

> signal → template → worked examples → toolkit

**Contents**

1. **The signal**
2. **Monotonic stack** — nearest greater / smaller
3. **Monotonic stack** — largest rectangle in a histogram
4. **Monotonic queue** — sliding window maximum
5. **When to use & cheat-sheet**

## 1. The signal — discard what can't win

Reach for a monotonic structure when a problem asks, for each element, about the **nearest element that is greater or smaller** (to the left or right), or the **max/min of every sliding window**. The brute force compares each element against the others — O(n²).

The insight: keep a stack/deque whose values stay **monotonic** (all increasing, or all decreasing). Before pushing a new value, **pop everything it dominates** — those popped elements have just found their answer (the new value), and they can never be the nearest-greater for anything further on. Each element is pushed and popped **at most once**, so the whole sweep is **O(n)**.

## 2. Monotonic stack — nearest greater / smaller

For "next greater element," keep a stack of indices whose values are **decreasing**. When a new value arrives, it's the answer for every smaller value waiting on top — pop them and record it:

In [1]:
def next_greater(a):
    res = [-1] * len(a)
    stack = []                           # indices; their values stay decreasing
    for i, x in enumerate(a):
        while stack and a[stack[-1]] < x:
            res[stack.pop()] = x         # x is the next-greater for everything it beats
        stack.append(i)
    return res

print("next_greater([2,1,2,4,3]):", next_greater([2, 1, 2, 4, 3]))

# the index version answers "how many steps until a greater value?" (daily temperatures):
def daily_temperatures(temps):
    res = [0] * len(temps)
    stack = []
    for i, t in enumerate(temps):
        while stack and temps[stack[-1]] < t:
            j = stack.pop()
            res[j] = i - j               # a distance, not a value
        stack.append(i)
    return res

print("daily_temperatures:", daily_temperatures([73, 74, 75, 71, 69, 72, 76, 73]))


next_greater([2,1,2,4,3]): [4, 2, 4, -1, -1]
daily_temperatures: [1, 1, 4, 2, 1, 1, 0, 0]


Flip the comparison (`>` instead of `<`) for *smaller*; iterate right-to-left for *previous* instead of *next*. Four variants — next/prev × greater/smaller — all the same template.

## 3. Monotonic stack — largest rectangle in a histogram

The pattern's showcase. For each bar, the largest rectangle using it as the limiting height extends left and right until a **shorter** bar stops it — exactly the "nearest smaller on each side" a monotonic stack finds. Keep indices of **increasing** heights; when a shorter bar arrives, pop and settle each taller bar's rectangle (its width is bounded by the new bar on the right and whatever remains on the stack to the left):

In [2]:
def largest_rectangle(heights):
    stack = []                           # indices; their heights stay increasing
    best = 0
    for i, h in enumerate(heights + [0]):        # a trailing 0 sentinel flushes the stack
        while stack and heights[stack[-1]] > h:
            height = heights[stack.pop()]
            left = stack[-1] if stack else -1
            width = i - left - 1                 # bounded by the new bar (right) + the stack (left)
            best = max(best, height * width)
        stack.append(i)
    return best

print("largest_rectangle([2,1,5,6,2,3]):", largest_rectangle([2, 1, 5, 6, 2, 3]))   # 10


largest_rectangle([2,1,5,6,2,3]): 10


The same skeleton solves "maximal rectangle" in a binary matrix (run it per row, treating column run-lengths as histogram heights) and shows up in "trapping rain water.

## 4. Monotonic queue — sliding window maximum

A monotonic **deque** gives the max (or min) of every size-k window in **O(n)** — the piece the sliding-window notebook deferred. Hold indices in **decreasing** value order so the front is always the window's max. Each step: pop smaller values from the back (the newcomer dominates them), then drop the front if it has slid out of the window:

In [3]:
from collections import deque

def max_sliding_window(a, k):
    dq = deque()                         # indices; their values stay decreasing
    out = []
    for i, x in enumerate(a):
        while dq and a[dq[-1]] <= x:
            dq.pop()                     # x dominates these -> they can never be a future max
        dq.append(i)
        if dq[0] <= i - k:               # the front index has left the window
            dq.popleft()
        if i >= k - 1:                   # window is full -> its max is the front
            out.append(a[dq[0]])
    return out

print("max_sliding_window([1,3,-1,-3,5,3,6,7], 3):", max_sliding_window([1, 3, -1, -3, 5, 3, 6, 7], 3))


max_sliding_window([1,3,-1,-3,5,3,6,7], 3): [3, 3, 5, 5, 6, 7]


This is why `collections.deque` (the linked-lists notebook's block-based doubly linked list) is the right container — O(1) at *both* ends, which a plain `list` can't offer at the front.

## 5. When to use & cheat-sheet

| The problem asks for… | Use |
|---|---|
| nearest greater / smaller element (either side) | monotonic **stack** |
| span / distance to the next bigger value | monotonic stack (store indices) |
| largest rectangle / max area under bars | monotonic stack |
| max / min of every sliding window | monotonic **deque** |

**Python toolkit:** a plain `list` is a fine monotonic *stack* (`append`/`pop` at the end are O(1)); use `collections.deque` for the monotonic *queue* because you pop from **both** ends. Store **indices**, not values, whenever you need positions, distances, or window bounds.

**The invariant to hold in your head:** *an element is useless the moment a later element dominates it.* Popping those eagerly is what makes each element O(1) amortized.

| Problem | Structure | Time | vs brute force |
|---|---|:---:|:---:|
| Next / prev greater / smaller | monotonic stack | O(n) | O(n²) |
| Largest rectangle in histogram | monotonic stack | O(n) | O(n²) |
| Sliding window max / min | monotonic deque | O(n) | O(n·k) |